<a href="https://colab.research.google.com/github/KickGosu/AI-Trainning-Practice/blob/main/B%E1%BA%A3n_sao_c%E1%BB%A7a_Bai1_linearregession_sms_callmobile_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bài 1: Phân tích dữ liệu CDR viễn thông và Linear Regression

Notebook này hướng dẫn các bước cơ bản để phân tích dữ liệu CDR (Call Detail Records) viễn thông: đọc dữ liệu từ nhiều file CSV, khám phá dữ liệu (EDA), xử lý missing values, phân tích hoạt động theo giờ, so sánh cuộc gọi trong nước và quốc tế, và trực quan hóa dữ liệu với matplotlib/seaborn. Cuối bài, ta xây dựng hai mô hình Linear Regression: (1) dự đoán SMS đi từ nhiều đặc trưng hoạt động, và (2) dự đoán lưu lượng cuộc gọi từ dữ liệu SMS.

**Thời lượng dự kiến:** 1h45 - 2h

### Cấu trúc notebook

| Phần | Nội dung | Mục tiêu |
|------|----------|----------|
| **A** | Thiết lập môi trường | Cài đặt thư viện, import, kết nối dữ liệu, tổ chức thư mục output |
| **B** | Khám phá dữ liệu CDR (EDA) | Giới thiệu dataset, đọc file, kiểm tra cấu trúc, thống kê mô tả, trực quan hóa cơ bản |
| **C** | Xử lý và phân tích dữ liệu | Gộp nhiều file, xử lý missing values, phân tích theo giờ, so sánh ngày/đêm, trong nước/quốc tế, phân tích tương quan |
| **D** | Xây dựng mô hình dự đoán | Thử nghiệm 1: Dự đoán smsout từ nhiều đặc trưng; Thử nghiệm 2: Dự đoán total_calls từ total_sms; Train/val/test split, đánh giá, lưu & load mô hình |
| **E** | Tổng kết | Nhắc lại các kỹ năng đã thực hành |

Sau khi hoàn thành, người học sẽ nắm được: cách đọc và kiểm tra dữ liệu với pandas, xử lý missing values, phân tích dữ liệu theo thời gian, phân biệt cuộc gọi trong nước và quốc tế, tính toán tương quan giữa các loại hoạt động, và xây dựng mô hình hồi quy tuyến tính với sklearn.

---
## Phần A: Thiết lập môi trường

## A.1. Cài đặt thư viện

Cell này cài các thư viện phổ biến dùng xuyên suốt khóa học. Nếu môi trường đã có sẵn thư viện, bước này sẽ chạy rất nhanh.

In [1]:
import sys
import subprocess

packages = ["pandas", "numpy", "matplotlib", "seaborn", "scikit-learn", "joblib"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)

print("Đã sẵn sàng môi trường thực hành.")

Đã sẵn sàng môi trường thực hành.


## A.2. Import và cấu hình hiển thị

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

print("✅ Thư viện đã sẵn sàng!")

✅ Thư viện đã sẵn sàng!


## A.3. Kết nối dữ liệu

Notebook ưu tiên đọc dữ liệu từ Google Drive nếu chạy trên Colab. Khi chạy trong workspace hiện tại, notebook sẽ tự dùng thư mục `data/` cục bộ. Dữ liệu được tải từ Kaggle: https://www.kaggle.com/datasets/marcodena/mobile-phone-activity

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
LOCAL_DATA_ROOT = PROJECT_ROOT / "data"
DRIVE_DATA_ROOT = Path("/content/drive/MyDrive/CourseAI/data")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_ROOT = DRIVE_DATA_ROOT if DRIVE_DATA_ROOT.exists() else LOCAL_DATA_ROOT
except Exception:
    DATA_ROOT = LOCAL_DATA_ROOT

print("DATA_ROOT:", DATA_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DATA_ROOT: /content/drive/MyDrive/CourseAI/data


In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "outputs" / "Bai1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("OUTPUT_DIR:", OUTPUT_DIR)

OUTPUT_DIR: /content/notebooks/outputs/Bai1


---
## Phần B: Khám phá dữ liệu CDR (EDA)

## B.1. Giới thiệu bộ dữ liệu CDR

### Tổng quan
Bộ dữ liệu Mobile Phone Activity ghi lại một tuần Call Details Records (CDR) tại Milan, Ý.

### Mô tả bộ dữ liệu
Mỗi khi người dùng thực hiện tương tác viễn thông, một Trạm Gốc Vô Tuyến (RBS - Radio Base Station) được nhà mạng chỉ định để truyền tải thông tin qua mạng. Khi đó, một bản ghi CDR mới được tạo, ghi lại thời gian tương tác và RBS đã xử lý nó.

Các hoạt động có trong bộ dữ liệu:

- Nhận SMS (received SMS - `smsin`)
- Gửi SMS (sent SMS - `smsout`)
- Cuộc gọi đến (incoming calls - `callin`)
- Cuộc gọi đi (outgoing calls - `callout`)
- Hoạt động Internet (`internet`): Được tạo mỗi khi người dùng bắt đầu hoặc kết thúc kết nối Internet. Trong cùng một kết nối, một CDR được tạo nếu kết nối kéo dài hơn 15 phút hoặc người dùng truyền tải hơn 5 MB.

Dữ liệu được tổng hợp theo không gian trong lưới ô vuông. Khu vực gồm một lưới phủ 1,000 ô vuông với kích thước khoảng 235×235 mét mỗi ô.

Lưới này được chiếu theo chuẩn WGS84 (EPSG:4326). Tham khảo bài báo gốc: http://go.nature.com/2fcOX5E

Dữ liệu cung cấp CellID, CountryCode và tất cả các hoạt động viễn thông kể trên, được tổng hợp mỗi 60 phút.

Tìm hiểu thêm tại: https://www.kaggle.com/datasets/marcodena/mobile-phone-activity

## B.2. Tải bộ dữ liệu từ Kaggle

Bộ dữ liệu được tải từ Kaggle. Dưới đây là hướng dẫn kết nối Kaggle với Google Colab để tải dữ liệu tự động (nếu chạy trên Colab). Khi chạy local, dữ liệu đã được tải sẵn trong thư mục `data/`.

### Cách kết nối Kaggle với Google Colab (nếu dùng Colab)

In [ ]:
# Phương pháp 1 (an toàn hơn, khuyên dùng)
# Lưu trữ thông tin xác thực trong Colab Secrets trước (xem hướng dẫn bên dưới)
# Sau đó bỏ comment dòng dưới để tải dữ liệu:
# KGAT_39aa04968be7161f62e3bcd0d4109b14


from google.colab import userdata
import os
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

!kaggle datasets download -d marcodena/mobile-phone-activity
!unzip -o mobile-phone-activity.zip -d data/

Dataset URL: https://www.kaggle.com/datasets/marcodena/mobile-phone-activity
License(s): DbCL-1.0
mobile-phone-activity.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  mobile-phone-activity.zip
  inflating: data/ISTAT_census_variables_2011.csv  
  inflating: data/Italian_provinces.geojson  
  inflating: data/mi-to-provinces-2013-11-01.csv  
  inflating: data/mi-to-provinces-2013-11-02.csv  
  inflating: data/mi-to-provinces-2013-11-03.csv  
  inflating: data/mi-to-provinces-2013-11-04.csv  
  inflating: data/mi-to-provinces-2013-11-05.csv  
  inflating: data/mi-to-provinces-2013-11-06.csv  
  inflating: data/mi-to-provinces-2013-11-07.csv  
  inflating: data/milano-grid.geojson  
  inflating: data/sms-call-internet-mi-2013-11-01.csv  
  inflating: data/sms-call-internet-mi-2013-11-02.csv  
  inflating: data/sms-call-internet-mi-2013-11-03.csv  
  inflating: data/sms-call-internet-mi-2013-11-04.csv  
  inflating: data/sms-call-internet-mi

## B.3. Quan sát thư mục dữ liệu

Trước khi đọc dữ liệu, ta nên xem trong thư mục có những file nào. Đây là thói quen quan trọng khi nhận một dataset mới.

### Câu hỏi khởi động:

- Có tổng cộng bao nhiêu file?
- Bạn nhận xét gì về tên file?
- Tại sao có 7 file cho mỗi loại SMS/calls/internet?
- File `mi-to-provinces` có thể có ý nghĩa gì?
- File nào có dung lượng lớn nhất?

In [ ]:
# Liệt kê tất cả file dữ liệu trong thư mục data
data_path = DATA_ROOT / "mobile-phone-activity"
if not data_path.exists():
    data_path = DATA_ROOT

files = sorted([f for f in os.listdir(data_path) if f.endswith('.csv') or f.endswith('.geojson')])

print('*' * 70)
print(f"Nội dung thư mục dữ liệu: Tìm thấy {len(files)} file")
print('*' * 70)
for i, f in enumerate(files, 1):
    size = os.path.getsize(data_path / f) / 1024**2
    print(f"  {i:2d}. {f:50s} ({size:6.2f} MB)")

**********************************************************************
Nội dung thư mục dữ liệu: Tìm thấy 0 file
**********************************************************************


### Nhận xét ban đầu:
- Có **7 file** `sms-call-internet-mi-2013-11-0X.csv` tương ứng với 7 ngày trong tuần (01-07/11/2013).
- Có **7 file** `mi-to-provinces-2013-11-0X.csv` ghi lại luồng di chuyển giữa các tỉnh.
- File `.geojson` chứa thông tin bản đồ (lưới Milano và bản đồ các tỉnh Ý).
- File `ISTAT_census_variables_2011.csv` chứa dữ liệu thống kê dân số.

## B.4. Đọc và khám phá dữ liệu một ngày

Bắt đầu bằng việc đọc dữ liệu của **một ngày** (01-11-2013) để làm quen với cấu trúc trước khi gộp nhiều ngày. Đây là cách tiếp cận từng bước: hiểu một đơn vị nhỏ trước khi mở rộng.

In [ ]:
# Đọc file dữ liệu SMS/call/internet ngày 01-11-2013
cdr_df = pd.read_csv(data_path / "sms-call-internet-mi-2013-11-01.csv")
print("Đã đọc file dữ liệu ngày 01-11-2013")
cdr_df.info()

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/CourseAI/data/sms-call-internet-mi-2013-11-01.csv'

### Câu hỏi:
- Mỗi dòng trong dữ liệu đại diện cho điều gì?
- Có bao nhiêu cột? Mỗi cột có ý nghĩa gì?
- Kiểu dữ liệu của từng cột là gì?
- Dung lượng bộ nhớ dataset chiếm bao nhiêu?

In [ ]:
# Xem 10 dòng đầu tiên
print("\n10 dòng đầu tiên:")
cdr_df.head(10)

### Nhận xét từ việc xem 10 dòng đầu:
- Mỗi dòng đại diện cho hoạt động viễn thông tại một **ô lưới (CellID)**, vào một **giờ cụ thể (datetime)**, với một **mã quốc gia (countrycode)** cụ thể.
- Các cột hoạt động (`smsin`, `smsout`, `callin`, `callout`, `internet`) có nhiều giá trị **NaN** — không phải ô lưới nào cũng có tất cả các loại hoạt động với tất cả các quốc gia.
- `countrycode=39` là mã nước Ý (gọi trong nước), `countrycode=0` có thể là tổng hợp, các mã khác là quốc tế.
- `countrycode=33` là Pháp — láng giềng với Ý.

## B.5. Khám phá cấu trúc dữ liệu

Kiểm tra các thông tin cơ bản: kích thước, tên cột, kiểu dữ liệu và thống kê mô tả.

In [ ]:
# Khám phá cơ bản
print(f"\nKích thước tập dữ liệu: {cdr_df.shape[0]:,} dòng × {cdr_df.shape[1]} cột")

print(f"\nCác cột:")
for col in cdr_df.columns:
    print(f"   - {col}")

print("\nKiểu dữ liệu:")
print(cdr_df.dtypes)

In [ ]:
# Thống kê mô tả
print(f"\nThống kê mô tả:")
cdr_df.describe()

### Một số quan sát từ thống kê mô tả:
- `CellID` trải từ 1 đến 10,000 — đúng như mô tả 1,000 ô lưới.
- `countrycode` có giá trị max rất lớn (97,259) — cần kiểm tra thêm.
- `smsout` có ít giá trị non-null nhất (~469K) so với `smsin` (~806K) — nghĩa là nhiều bản ghi không có dữ liệu SMS đi.
- `callin` chỉ có ~484K giá trị non-null — có thể các cuộc gọi đến ít được ghi nhận hơn.
- Giá trị `internet` có độ lệch rất lớn (std=342 so với mean=102), max lên đến 27,774.

## B.6. Phân tích tương quan tổng thể

Tính ma trận tương quan giữa các cột số để xem mối quan hệ giữa các loại hoạt động.

In [ ]:
# Ma trận tương quan
corr_matrix = cdr_df.corr()
corr_matrix

In [ ]:
# Trực quan hóa ma trận tương quan bằng heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm',
            center=0, square=True, linewidths=1,
            cbar_kws={"shrink": 0.8})
plt.title('Ma trận tương quan giữa các cột dữ liệu CDR', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

### Nhận xét về tương quan:
- **`callin` và `callout`** có tương quan rất mạnh (0.969) — cuộc gọi đến và đi thường đi cùng nhau.
- **`smsin` và `callin`** có tương quan cao (0.836) — SMS đến tương quan mạnh với cuộc gọi đến.
- **`smsin` và `internet`** có tương quan cao (0.865) — hoạt động Internet và SMS đến liên quan chặt chẽ.
- **`smsin` và `smsout`** có tương quan khá (0.706) — nhưng không hoàn toàn đồng nhất.
- **`CellID`** có tương quan yếu với các hoạt động — vị trí địa lý không quyết định mạnh mức độ hoạt động.
- **`countrycode`** có tương quan âm nhẹ với tất cả các hoạt động.

## B.7. Trực quan hóa mối quan hệ giữa các cặp biến

Sử dụng `jointplot` và `lmplot` để xem chi tiết mối quan hệ giữa các cặp biến quan trọng. Đây là các biểu đồ có trong thử nghiệm gốc, giúp phát hiện insight trực quan.

In [ ]:
# Jointplot: smsin vs smsout
sns.jointplot(data=cdr_df, x="smsin", y="smsout", kind='scatter', alpha=0.3, height=8)
plt.suptitle('Mối quan hệ giữa SMS đến và SMS đi', y=1.02, fontweight='bold')
plt.show()

### Câu hỏi:
- Biểu đồ jointplot cho thấy điều gì về mối quan hệ smsin-smsout?
- Mật độ điểm tập trung ở đâu? Có điểm ngoại lai không?

In [ ]:
# Jointplot: callin vs callout
sns.jointplot(data=cdr_df, x="callin", y="callout", kind='scatter', alpha=0.3, height=8)
plt.suptitle('Mối quan hệ giữa Cuộc gọi đến và Cuộc gọi đi', y=1.02, fontweight='bold')
plt.show()

### Nhận xét:
- `callin` và `callout` có mối quan hệ gần như tuyến tính hoàn hảo — đây là cặp có tương quan cao nhất (0.969).
- Dữ liệu tập trung ở vùng giá trị thấp, với một số điểm văng ra xa (outlier).

In [ ]:
# Lmplot: smsin vs callin - kèm đường hồi quy
sns.lmplot(data=cdr_df, x="smsin", y="callin", height=7,
           scatter_kws={'alpha': 0.2, 's': 10}, line_kws={'color': 'red', 'linewidth': 2})
plt.title('Đường hồi quy: SMS đến vs Cuộc gọi đến', fontweight='bold')
plt.show()

In [ ]:
# Lmplot: callin vs callout - kèm đường hồi quy
sns.lmplot(data=cdr_df, x="callin", y="callout", height=7,
           scatter_kws={'alpha': 0.2, 's': 10}, line_kws={'color': 'red', 'linewidth': 2})
plt.title('Đường hồi quy: Cuộc gọi đến vs Cuộc gọi đi', fontweight='bold')
plt.show()

### Câu hỏi:
- Đường hồi quy có khớp tốt với dữ liệu không?
- Từ các biểu đồ trên, bạn dự đoán biến nào sẽ là predictor tốt nhất cho `smsout`?

## B.8. Pairplot tổng quan tất cả các biến

`pairplot` của seaborn vẽ tất cả các cặp quan hệ giữa các biến số trong dataset. Lưu ý: với dataset lớn, `pairplot` có thể chạy chậm. Ta có thể sample một phần dữ liệu để tăng tốc.

In [ ]:
# Sample 5000 dòng để vẽ pairplot (tránh quá tải)
df_sample = cdr_df.dropna().sample(n=5000, random_state=42)
print(f"Kích thước mẫu cho pairplot: {df_sample.shape}")

sns.pairplot(df_sample[['smsin', 'smsout', 'callin', 'callout', 'internet']],
             diag_kind='kde', plot_kws={'alpha': 0.3, 's': 10})
plt.suptitle('Pairplot các biến hoạt động viễn thông (mẫu 5,000 dòng)', y=1.02, fontweight='bold')
plt.show()

## B.9. Tập trung phân tích một ô lưới (CID=1)

Để hiểu sâu hơn về cấu trúc dữ liệu, ta phân tích chi tiết một ô lưới cụ thể. Đây là kỹ thuật "zoom in" quan trọng trong EDA: hiểu một đơn vị nhỏ trước khi khái quát hóa.

In [ ]:
# Tìm hiểu số lượng ô lưới
print(f"Tổng số bản ghi: {len(cdr_df):,}")
print(f"Số ô lưới duy nhất (CellID): {cdr_df['CellID'].nunique():,}")
print(f"Số mã quốc gia duy nhất (countrycode): {cdr_df['countrycode'].nunique():,}")

In [ ]:
# Lọc dữ liệu của CID = 1
sample_cid = 1
single_cid = cdr_df[cdr_df['CellID'] == sample_cid].copy()
print(f"Dữ liệu của ô lưới CID={sample_cid}: {single_cid.shape[0]} dòng")
single_cid.head(10)

In [ ]:
# Xem phân bố countrycode trong CID=1
print(f"Các mã quốc gia xuất hiện trong CID={sample_cid}:")
single_cid['countrycode'].value_counts().head(20)

### Câu hỏi:
- Tại CID=1, countrycode nào xuất hiện nhiều nhất?
- Tại sao có nhiều countrycode khác nhau trong cùng một ô lưới?
- Mỗi countrycode đại diện cho điều gì? (Gợi ý: countrycode=39 là Ý)

In [ ]:
# Phân tích hoạt động SMS đi tại CID=1
sms_out = single_cid['smsout'].notna()
sms_outgoing = single_cid[sms_out].copy()
print(f"Số dòng có dữ liệu SMS đi tại CID=1: {sms_outgoing.shape[0]}")

# Chuyển đổi datetime và trích xuất giờ
sms_outgoing['datetime'] = pd.to_datetime(sms_outgoing['datetime'])
sms_outgoing['hour'] = sms_outgoing['datetime'].dt.hour
sms_outgoing[['datetime', 'hour', 'smsout']].head(10)

## B.10. Trực quan hóa mẫu hình SMS theo giờ cho một ô lưới

Đây là một trong những trực quan hóa quan trọng để tìm insight về hành vi người dùng.

In [ ]:
# Vẽ biểu đồ SMS đi theo giờ cho CID=1
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Biểu đồ đường
axes[0].plot(sms_outgoing['hour'], sms_outgoing['smsout'],
             marker='o', linewidth=1.5, markersize=6, color='steelblue')
axes[0].set_title(f'Mẫu hình SMS đi theo giờ - CID={sample_cid}', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Giờ trong ngày')
axes[0].set_ylabel('Số lượng SMS đi')
axes[0].set_xticks(range(0, 24, 2))
axes[0].grid(True, alpha=0.3)

# Biểu đồ cột
hourly_sms = sms_outgoing.groupby('hour')['smsout'].mean()
axes[1].bar(hourly_sms.index, hourly_sms.values, color='chocolate', alpha=0.8)
axes[1].set_title(f'SMS đi trung bình theo giờ - CID={sample_cid}', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Giờ trong ngày')
axes[1].set_ylabel('SMS đi trung bình')
axes[1].set_xticks(range(0, 24, 2))
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### Câu hỏi thực hành:
- Hãy chọn một ô lưới khác (CID=100, CID=500, CID=1000) và vẽ biểu đồ tương tự. Mẫu hình có khác biệt không?
- Giờ cao điểm SMS đi tại CID=1 là giờ nào?
- Bạn có nhận thấy mẫu hình nào đặc trưng của hành vi con người?

---
## Phần C: Xử lý và phân tích dữ liệu

## C.1. Bài tập 1: Tải và gộp dữ liệu

### NHIỆM VỤ 1: Tải và gộp dữ liệu

1. Tải 3 file dữ liệu hoạt động SMS/call/internet (sms-call-internet-mi-2013-11-02.csv, sms-call-internet-mi-2013-11-04.csv, sms-call-internet-mi-2013-11-06.csv) và gộp chúng lại.

2. Tạo một tập dữ liệu sạch, sẵn sàng cho phân tích.

#### Yêu cầu:
- Tải cả 3 file hoạt động
- Thêm cột ngày và giờ
- Gộp vào một dataframe duy nhất
- Xử lý missing values (Gợi ý: Điền bằng giá trị trung bình)
- Thêm cột tổng hợp (total_sms, total_calls, total_internet)

#### Câu hỏi (50 điểm):
- Có tổng cộng bao nhiêu bản ghi trên cả 3 tập dữ liệu?
- Có bao nhiêu ô lưới (CellID) duy nhất?
- Có bao nhiêu mã quốc gia (countrycode) xuất hiện trong dữ liệu?
- Có missing values không? Nếu có, điền missing/NaN bằng giá trị trung bình của mỗi cột. Cột nào có nhiều missing values nhất? Bạn đã chỉnh sửa bao nhiêu bản ghi?
- Giờ cao điểm phổ biến nhất trên tất cả các ô lưới là gì? Giờ nào có hoạt động thấp nhất? Tính toán và báo cáo: mean, median, std, min, max cho tổng số cuộc gọi theo giờ.
- Tỷ lệ phần trăm tổng hoạt động diễn ra vào ban ngày (6h-20h) so với ban đêm (20h-6h)?
- Cuộc gọi quốc tế có diễn ra vào các khung giờ khác với cuộc gọi trong nước không?
- Sử dụng numpy để thực hiện so sánh thống kê giữa các điều kiện khác nhau. Cuộc gọi quốc tế có xu hướng đến hay đi nhiều hơn? Có tương quan giữa khối lượng SMS và khối lượng Cuộc gọi ở cấp độ ô lưới không?

## C.2. Tải và gộp 3 file dữ liệu hoạt động

Tải 3 file: sms-call-internet-mi-2013-11-02.csv, sms-call-internet-mi-2013-11-04.csv, sms-call-internet-mi-2013-11-06.csv và gộp chúng lại thành một dataframe duy nhất.

In [ ]:
# Tải từng file và kiểm tra cấu trúc
print("="*70)
print("ĐỌC 3 FILE DỮ LIỆU HOẠT ĐỘNG")
print("="*70)

sms_call_internet_02_df = pd.read_csv(data_path / "sms-call-internet-mi-2013-11-02.csv")
print(f"\nFile 02/11: {sms_call_internet_02_df.shape[0]:,} dòng")
print(sms_call_internet_02_df.info())

In [ ]:
sms_call_internet_04_df = pd.read_csv(data_path / "sms-call-internet-mi-2013-11-04.csv")
print(f"\nFile 04/11: {sms_call_internet_04_df.shape[0]:,} dòng")
print(sms_call_internet_04_df.info())

In [ ]:
sms_call_internet_06_df = pd.read_csv(data_path / "sms-call-internet-mi-2013-11-06.csv")
print(f"\nFile 06/11: {sms_call_internet_06_df.shape[0]:,} dòng")
print(sms_call_internet_06_df.info())

### Gộp vào một dataframe duy nhất

Sử dụng `pd.concat` để nối 3 dataframe theo chiều dọc (axis=0). Tham số `ignore_index=True` giúp đánh lại chỉ số từ 0 sau khi gộp.

In [ ]:
# Gộp 3 dataframe vào một dataframe duy nhất
combined_df = pd.concat(
    [sms_call_internet_02_df, sms_call_internet_04_df, sms_call_internet_06_df],
    ignore_index=True
)
print(f"Tổng số dòng sau khi gộp: {combined_df.shape[0]:,}")
print(f"Số cột: {combined_df.shape[1]}")
combined_df.head(3)

## C.3. Thêm cột ngày và giờ

Chuyển đổi cột `datetime` từ kiểu `object` sang `datetime64` và trích xuất ngày, giờ vào cột mới. Đây là bước quan trọng cho phân tích theo thời gian.

In [ ]:
# Chuyển đổi datetime và trích xuất ngày, giờ
combined_df['datetime'] = pd.to_datetime(combined_df['datetime'])
combined_df['date'] = combined_df['datetime'].dt.date
combined_df['hour'] = combined_df['datetime'].dt.hour
combined_df['time'] = combined_df['datetime'].dt.time

print("Đã thêm các cột: date, hour, time")
combined_df[['datetime', 'date', 'hour', 'time']].head()

In [ ]:
# Xem các ngày có trong dữ liệu
print("Các ngày trong dữ liệu đã gộp:")
for d in sorted(combined_df['date'].unique()):
    count = len(combined_df[combined_df['date'] == d])
    print(f"  {d}: {count:,} bản ghi")

### Có tổng cộng bao nhiêu bản ghi trên cả 3 tập dữ liệu?

In [ ]:
# Tổng số bản ghi (dòng) trong tập dữ liệu đã gộp
print("Tổng số bản ghi:", combined_df.shape[0])

### Có bao nhiêu ô lưới (CellID) duy nhất?

In [ ]:
# Số lượng ô lưới duy nhất
print("Số ô lưới duy nhất (CellID):", combined_df['CellID'].nunique())

### Có bao nhiêu mã quốc gia (countrycode) xuất hiện trong dữ liệu?

In [ ]:
# Số lượng mã quốc gia duy nhất
print("Số mã quốc gia duy nhất trong dữ liệu:", combined_df['countrycode'].nunique())

In [ ]:
# Xem phân bố top 20 countrycode
country_counts = combined_df['countrycode'].value_counts().head(20)
print("Top 20 mã quốc gia xuất hiện nhiều nhất:")
country_counts

## C.4. Kiểm tra và xử lý missing values

### Có missing values không?
- Nếu có, điền missing/NaN bằng giá trị trung bình của mỗi cột.
- Cột nào có nhiều missing values nhất?
- Bạn đã chỉnh sửa bao nhiêu bản ghi?

In [ ]:
# Kiểm tra số lượng missing values trong mỗi cột
print("Số lượng missing values trong mỗi cột:")
missing_before = combined_df.isna().sum()
missing_before

In [ ]:
# Trực quan hóa missing values
missing_pct = (combined_df.isna().sum() / len(combined_df)) * 100
missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)

plt.figure(figsize=(10, 5))
bars = plt.bar(missing_pct.index, missing_pct.values, color='chocolate', alpha=0.8)
plt.title('Tỷ lệ missing values theo cột (%)', fontweight='bold', fontsize=13)
plt.xlabel('Cột')
plt.ylabel('Tỷ lệ thiếu dữ liệu (%)')
plt.xticks(rotation=45)
for bar, pct in zip(bars, missing_pct.values):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
             f'{pct:.1f}%', ha='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

### Xử lý missing values: Điền bằng giá trị trung bình

Đây là phương pháp đơn giản nhất: thay thế mỗi giá trị NaN bằng giá trị trung bình của cột đó. `fillna()` của pandas thực hiện việc này một cách hiệu quả. Phương pháp này phù hợp khi dữ liệu thiếu ngẫu nhiên và không quá nhiều.

In [ ]:
print('Số lượng missing values TRƯỚC khi điền:')
print(combined_df[['smsin', 'smsout', 'callin', 'callout', 'internet']].isnull().sum())

# Điền missing values bằng giá trị trung bình của mỗi cột
modified_count = 0

for col in ['smsin', 'smsout', 'callin', 'callout', 'internet']:
    original_nans = combined_df[col].isnull().sum()
    combined_df[col] = combined_df[col].fillna(combined_df[col].mean())
    modified_count += original_nans

print(f"\nSố lượng bản ghi đã được điền (tổng số NaN đã thay thế): {modified_count:,}")
print(f"Số dòng bị ảnh hưởng: {combined_df[['smsin', 'smsout', 'callin', 'callout', 'internet']].isna().any(axis=1).sum():,}")

print('\nSố lượng missing values SAU khi điền:')
print(combined_df[['smsin', 'smsout', 'callin', 'callout', 'internet']].isnull().sum())

### Kết luận xử lý missing:
- Cột **smsout** có nhiều missing values nhất, với hơn 5 triệu giá trị thiếu.
- Cột **callin** cũng có tỷ lệ thiếu rất cao.
- Tổng cộng đã điền hơn 21 triệu giá trị bằng phương pháp mean imputation.
- Đây là cách xử lý đơn giản, phù hợp cho bài tập đầu tiên. Trong thực tế, ta có thể cân nhắc các phương pháp phức tạp hơn như KNN imputation, MICE, hoặc xây dựng model riêng để dự đoán missing values.

## C.5. Thêm cột tổng hợp

Tạo các cột tổng hợp để phân tích: tổng SMS, tổng cuộc gọi và tổng Internet. Các cột này giúp đơn giản hóa việc phân tích tổng quan.

In [ ]:
# Thêm các cột tổng hợp
combined_df['total_sms'] = combined_df['smsin'] + combined_df['smsout']
combined_df['total_calls'] = combined_df['callin'] + combined_df['callout']
combined_df['total_internet'] = combined_df['internet']

print("Đã thêm 3 cột tổng hợp: total_sms, total_calls, total_internet")
combined_df[['smsin', 'smsout', 'total_sms', 'callin', 'callout', 'total_calls', 'internet', 'total_internet']].head()

## C.6. Phân tích hoạt động theo giờ

Phân tích mẫu hình hoạt động theo từng giờ trong ngày để tìm ra giờ cao điểm và thấp điểm.

In [ ]:
# Tính tổng hoạt động theo giờ
hourly_activity = combined_df.groupby('hour')[['total_sms', 'total_calls', 'total_internet']].sum().reset_index()
hourly_activity['total_activity_hourly'] = (hourly_activity['total_sms'] +
                                              hourly_activity['total_calls'] +
                                              hourly_activity['total_internet'])
print("Đã tổng hợp hoạt động theo giờ:")
hourly_activity

### Giờ cao điểm:

In [ ]:
# Tìm giờ có hoạt động cao nhất
peak_hour = hourly_activity.loc[hourly_activity['total_activity_hourly'].idxmax()]
print(f"Giờ cao điểm: {int(peak_hour['hour'])}h - Tổng hoạt động: {peak_hour['total_activity_hourly']:,.2f}")

# Tìm top 5 giờ cao điểm
print("\nTop 5 giờ cao điểm:")
hourly_activity_sorted = hourly_activity.sort_values('total_activity_hourly', ascending=False)
hourly_activity_sorted[['hour', 'total_activity_hourly']].head(5)

### Giờ có hoạt động thấp nhất:

In [ ]:
# Tìm giờ có hoạt động thấp nhất
lowest_hour = hourly_activity.loc[hourly_activity['total_activity_hourly'].idxmin()]
print(f"Giờ thấp điểm: {int(lowest_hour['hour'])}h - Tổng hoạt động: {lowest_hour['total_activity_hourly']:,.2f}")

In [ ]:
# BIỂU ĐỒ 1: Tổng hoạt động theo giờ - cột chồng
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Biểu đồ cột tổng
axes[0].bar(hourly_activity['hour'], hourly_activity['total_activity_hourly'],
            color='chocolate', alpha=0.85)
axes[0].set_xlabel('Giờ trong ngày', fontsize=12)
axes[0].set_ylabel('Tổng hoạt động', fontsize=12)
axes[0].set_title('Tổng hoạt động theo giờ trong ngày', fontweight='bold', fontsize=13)
axes[0].set_xticks(range(0, 24))
axes[0].grid(axis='y', linestyle='--', alpha=0.3)
# Đánh dấu giờ cao điểm và thấp điểm
axes[0].annotate(f'Cao điểm: {int(peak_hour["hour"])}h',
                 xy=(peak_hour['hour'], peak_hour['total_activity_hourly']),
                 xytext=(peak_hour['hour'] + 2, peak_hour['total_activity_hourly'] * 0.95),
                 arrowprops=dict(arrowstyle='->', color='red'), fontweight='bold', color='red')
axes[0].annotate(f'Thấp điểm: {int(lowest_hour["hour"])}h',
                 xy=(lowest_hour['hour'], lowest_hour['total_activity_hourly']),
                 xytext=(lowest_hour['hour'] + 2, lowest_hour['total_activity_hourly'] * 1.5),
                 arrowprops=dict(arrowstyle='->', color='blue'), fontweight='bold', color='blue')

# Biểu đồ đường cho từng loại hoạt động
axes[1].plot(hourly_activity['hour'], hourly_activity['total_sms'],
             marker='o', linewidth=2, markersize=6, label='SMS', color='steelblue')
axes[1].plot(hourly_activity['hour'], hourly_activity['total_calls'],
             marker='s', linewidth=2, markersize=6, label='Cuộc gọi', color='chocolate')
axes[1].plot(hourly_activity['hour'], hourly_activity['total_internet'],
             marker='^', linewidth=2, markersize=6, label='Internet', color='seagreen')
axes[1].set_xlabel('Giờ trong ngày', fontsize=12)
axes[1].set_ylabel('Tổng hoạt động theo loại', fontsize=12)
axes[1].set_title('Mẫu hình hoạt động theo giờ - Phân theo loại', fontweight='bold', fontsize=13)
axes[1].set_xticks(range(0, 24))
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Thống kê mô tả cho tổng số cuộc gọi theo giờ

Tính toán và báo cáo: mean, median, std, min, max cho total_calls theo từng giờ.

In [ ]:
print("Thống kê mô tả cho tổng số cuộc gọi (total_calls) theo giờ:")
calls_by_hour_stats = combined_df.groupby('hour')['total_calls'].agg(
    ['mean', 'median', 'std', 'min', 'max', 'count']
)
calls_by_hour_stats

In [ ]:
# BIỂU ĐỒ 2: Boxplot total_calls theo giờ
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Boxplot
combined_df.boxplot(column='total_calls', by='hour', ax=axes[0], showfliers=False)
axes[0].set_title('Phân bố total_calls theo giờ (boxplot, không outlier)', fontweight='bold')
axes[0].set_xlabel('Giờ trong ngày')
axes[0].set_ylabel('Tổng số cuộc gọi')
axes[0].set_xticklabels(range(0, 24))

# Mean ± std
axes[1].plot(calls_by_hour_stats.index, calls_by_hour_stats['mean'],
             marker='o', linewidth=2, color='steelblue', label='Mean')
axes[1].fill_between(calls_by_hour_stats.index,
                      calls_by_hour_stats['mean'] - calls_by_hour_stats['std'],
                      calls_by_hour_stats['mean'] + calls_by_hour_stats['std'],
                      alpha=0.3, color='steelblue', label='±1 Std')
axes[1].set_title('Mean ± Std của total_calls theo giờ', fontweight='bold')
axes[1].set_xlabel('Giờ trong ngày')
axes[1].set_ylabel('Tổng số cuộc gọi')
axes[1].set_xticks(range(0, 24, 2))
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('')
plt.tight_layout()
plt.show()

### Insight về hoạt động theo giờ:
- Giờ cao điểm thường rơi vào **khoảng 17h-19h** — thời điểm tan sở, mọi người sử dụng điện thoại nhiều.
- Giờ thấp điểm là **khoảng 3h-5h sáng** — thời gian mọi người ngủ.
- Mẫu hình chung: hoạt động tăng dần từ sáng, đạt đỉnh chiều tối, giảm mạnh vào ban đêm.
- Cả SMS, Cuộc gọi và Internet đều theo mẫu hình tương tự theo giờ.

## C.7. So sánh hoạt động ban ngày và ban đêm

Phân loại hoạt động thành hai khoảng thời gian: ban ngày (6h-20h) và ban đêm (20h-6h sáng hôm sau).

In [ ]:
# Phân loại mỗi giờ vào 'daytime' hoặc 'nighttime'
hourly_activity['time_period'] = hourly_activity['hour'].apply(
    lambda x: 'Ban ngày (6h-20h)' if 6 <= x < 20 else 'Ban đêm (20h-6h)'
)
print("Hoạt động theo giờ với phân loại ngày/đêm:")
hourly_activity[['hour', 'total_activity_hourly', 'time_period']]

In [ ]:
# Tính tổng hoạt động cho mỗi khoảng thời gian
time_period_activity = hourly_activity.groupby('time_period')['total_activity_hourly'].sum().reset_index()
overall_total_activity = time_period_activity['total_activity_hourly'].sum()
time_period_activity['percentage'] = (time_period_activity['total_activity_hourly'] / overall_total_activity) * 100

print(f"Tổng hoạt động toàn bộ: {overall_total_activity:,.2f}")
print("\nPhân bổ theo khoảng thời gian:")
time_period_activity

In [ ]:
# BIỂU ĐỒ 3: So sánh hoạt động ngày/đêm
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Pie chart
colors_pie = ['#ffcc80', '#5c6bc0']
axes[0].pie(time_period_activity['percentage'], labels=time_period_activity['time_period'],
            autopct='%1.1f%%', colors=colors_pie, startangle=90, explode=(0, 0.05),
            textprops={'fontsize': 12, 'fontweight': 'bold'})
axes[0].set_title('Tỷ lệ hoạt động: Ban ngày vs Ban đêm', fontweight='bold', fontsize=13)

# Bar chart
bars = axes[1].bar(time_period_activity['time_period'], time_period_activity['total_activity_hourly'],
                   color=colors_pie, alpha=0.85, edgecolor='black')
axes[1].set_title('Tổng hoạt động: Ban ngày vs Ban đêm', fontweight='bold', fontsize=13)
axes[1].set_ylabel('Tổng hoạt động')
axes[1].grid(axis='y', linestyle='--', alpha=0.3)
for bar, val, pct in zip(bars,
                         time_period_activity['total_activity_hourly'],
                         time_period_activity['percentage']):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + bar.get_height() * 0.01,
                 f'{val:,.0f}\n({pct:.1f}%)', ha='center', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()

### Nhận xét:
- **Khoảng 73-74%** lưu lượng diễn ra vào ban ngày (6h-20h).
- **Khoảng 26-27%** lưu lượng diễn ra vào ban đêm (20h-6h).
- Tỷ lệ này phản ánh nhịp sinh học tự nhiên của con người: hoạt động nhiều vào ban ngày, ít hơn vào ban đêm.
- Tỷ lệ 27% vào ban đêm vẫn khá đáng kể — có thể do một bộ phận dân số hoạt động về đêm, hoặc các kết nối Internet tự động (background data).

## C.8. So sánh cuộc gọi trong nước và quốc tế

Phân tích sự khác biệt giữa cuộc gọi trong nước (countrycode=39, Ý) và quốc tế (các countrycode khác).

In [ ]:
# Tạo cột phân loại trong nước / quốc tế
combined_df['call_type'] = np.where(combined_df['countrycode'] == 39, 'Trong nước', 'Quốc tế')
combined_df['sms_type'] = np.where(combined_df['countrycode'] == 39, 'Trong nước', 'Quốc tế')

print("Đã thêm cột 'call_type' và 'sms_type'")
# Kiểm tra tỷ lệ
print(f"\nPhân bố loại cuộc gọi:")
print(combined_df['call_type'].value_counts())
print(f"\nTỷ lệ %:")
print(combined_df['call_type'].value_counts(normalize=True) * 100)

### Cuộc gọi quốc tế có diễn ra vào các khung giờ khác với cuộc gọi trong nước không?

In [ ]:
# Nhóm dữ liệu cuộc gọi theo giờ và loại
hourly_calls = combined_df.groupby(['hour', 'call_type'])['total_calls'].sum().reset_index()
print("Khối lượng cuộc gọi theo giờ, nhóm theo loại:")
hourly_calls.head(10)

In [ ]:
# BIỂU ĐỒ 4: So sánh mẫu hình cuộc gọi trong nước và quốc tế theo giờ
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Biểu đồ đường so sánh trực tiếp
axes[0].plot(hourly_calls[hourly_calls['call_type'] == 'Trong nước']['hour'],
             hourly_calls[hourly_calls['call_type'] == 'Trong nước']['total_calls'],
             label='Cuộc gọi trong nước', marker='o', linestyle='-', linewidth=2,
             color='steelblue', markersize=6)
axes[0].plot(hourly_calls[hourly_calls['call_type'] == 'Quốc tế']['hour'],
             hourly_calls[hourly_calls['call_type'] == 'Quốc tế']['total_calls'],
             label='Cuộc gọi quốc tế', marker='s', linestyle='--', linewidth=2,
             color='chocolate', markersize=6)
axes[0].set_title('Khối lượng cuộc gọi theo giờ: Trong nước vs Quốc tế', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Giờ trong ngày')
axes[0].set_ylabel('Tổng khối lượng cuộc gọi')
axes[0].set_xticks(range(0, 24))
axes[0].legend(fontsize=11)
axes[0].grid(True, linestyle='--', alpha=0.5)

# Biểu đồ tỷ lệ phần trăm theo giờ
hourly_pivot = hourly_calls.pivot(index='hour', columns='call_type', values='total_calls').fillna(0)
hourly_pivot['total'] = hourly_pivot['Trong nước'] + hourly_pivot['Quốc tế']
hourly_pivot['% Trong nước'] = hourly_pivot['Trong nước'] / hourly_pivot['total'] * 100
hourly_pivot['% Quốc tế'] = hourly_pivot['Quốc tế'] / hourly_pivot['total'] * 100

axes[1].bar(hourly_pivot.index, hourly_pivot['% Trong nước'],
            label='% Trong nước', color='steelblue', alpha=0.8)
axes[1].bar(hourly_pivot.index, hourly_pivot['% Quốc tế'],
            bottom=hourly_pivot['% Trong nước'], label='% Quốc tế', color='chocolate', alpha=0.8)
axes[1].set_title('Tỷ lệ cuộc gọi trong nước/quốc tế theo giờ', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Giờ trong ngày')
axes[1].set_ylabel('Tỷ lệ (%)')
axes[1].set_xticks(range(0, 24))
axes[1].legend(fontsize=11)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### Nhận xét về mẫu hình cuộc gọi trong nước và quốc tế:
- Cả hai loại đều có mẫu hình tương tự theo giờ (tăng dần, đạt đỉnh chiều tối).
- Tỷ lệ cuộc gọi quốc tế **cao hơn** cuộc gọi trong nước ở hầu hết các khung giờ — điều này có thể do cấu trúc dữ liệu (nhiều countrycode quốc tế được ghi nhận cho mỗi CellID).
- Cuộc gọi quốc tế vẫn duy trì tỷ lệ cao vào ban đêm — có thể phản ánh sự khác biệt múi giờ giữa các quốc gia.

### Tỷ lệ phần trăm cuộc gọi quốc tế so với trong nước

In [ ]:
# Tính tổng số cuộc gọi theo loại
call_type_totals = combined_df.groupby('call_type')['total_calls'].sum().reset_index()
overall_total_calls = call_type_totals['total_calls'].sum()
call_type_totals['percentage'] = (call_type_totals['total_calls'] / overall_total_calls) * 100

print("Tổng khối lượng cuộc gọi theo loại:")
call_type_totals

### Tỷ lệ phần trăm SMS quốc tế so với trong nước

In [ ]:
# Tính tổng SMS theo loại
sms_type_totals = combined_df.groupby('sms_type')['total_sms'].sum().reset_index()
overall_total_sms = sms_type_totals['total_sms'].sum()
sms_type_totals['percentage'] = (sms_type_totals['total_sms'] / overall_total_sms) * 100

print("Tổng khối lượng SMS theo loại:")
sms_type_totals

In [ ]:
# BIỂU ĐỒ 5: So sánh trực quan tỷ lệ trong nước/quốc tế
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors = ['steelblue', 'chocolate']

# Cuộc gọi
axes[0].pie(call_type_totals['percentage'], labels=call_type_totals['call_type'],
            autopct='%1.1f%%', colors=colors, startangle=90, explode=(0, 0.05),
            textprops={'fontsize': 12, 'fontweight': 'bold'})
axes[0].set_title('Tỷ lệ Cuộc gọi: Trong nước vs Quốc tế', fontweight='bold', fontsize=13)

# SMS
axes[1].pie(sms_type_totals['percentage'], labels=sms_type_totals['sms_type'],
            autopct='%1.1f%%', colors=colors, startangle=90, explode=(0, 0.05),
            textprops={'fontsize': 12, 'fontweight': 'bold'})
axes[1].set_title('Tỷ lệ SMS: Trong nước vs Quốc tế', fontweight='bold', fontsize=13)

plt.tight_layout()
plt.show()

### Cuộc gọi quốc tế có xu hướng đến hay đi nhiều hơn?

In [ ]:
# Lọc cuộc gọi quốc tế và xác định xu hướng đến/đi
international_calls = combined_df[combined_df['call_type'] == 'Quốc tế']

total_international_callin = international_calls['callin'].sum()
total_international_callout = international_calls['callout'].sum()

print(f"Tổng cuộc gọi quốc tế đến:   {total_international_callin:,.2f}")
print(f"Tổng cuộc gọi quốc tế đi:    {total_international_callout:,.2f}")

if total_international_callout > 0:
    ratio = total_international_callin / total_international_callout
    print(f"Tỷ lệ cuộc gọi quốc tế đến / đi: {ratio:.2f}")
    if ratio > 1:
        print(f"→ Cuộc gọi quốc tế đến nhiều gấp {ratio:.1f} lần cuộc gọi quốc tế đi")
    else:
        print(f"→ Cuộc gọi quốc tế đi nhiều gấp {1/ratio:.1f} lần cuộc gọi quốc tế đến")

## C.9. Phân tích tương quan giữa các loại hoạt động ở cấp độ ô lưới

### Có tương quan giữa khối lượng SMS và khối lượng Cuộc gọi ở cấp độ ô lưới không?

In [ ]:
# Tổng hợp hoạt động theo ô lưới (CellID)
cell_activity = combined_df.groupby('CellID')[['total_sms', 'total_calls', 'total_internet']].sum().reset_index()

print(f"Số ô lưới: {len(cell_activity)}")
cell_activity.head()

In [ ]:
# Tính tương quan giữa các loại hoạt động ở cấp ô lưới
cell_corr = cell_activity[['total_sms', 'total_calls', 'total_internet']].corr()
print("Ma trận tương quan ở cấp độ ô lưới:")
cell_corr

In [ ]:
# BIỂU ĐỒ 6: Ma trận tương quan heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(cell_corr, annot=True, fmt='.4f', cmap='coolwarm',
            center=0.5, square=True, linewidths=2,
            cbar_kws={"shrink": 0.8})
plt.title('Ma trận tương quan giữa các loại hoạt động (cấp ô lưới)', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# BIỂU ĐỒ 7: Scatter plot SMS vs Cuộc gọi với đường hồi quy
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# SMS vs Calls
axes[0].scatter(cell_activity['total_sms'], cell_activity['total_calls'],
                alpha=0.4, s=15, color='steelblue')
axes[0].set_xlabel('Tổng SMS')
axes[0].set_ylabel('Tổng Cuộc gọi')
axes[0].set_title(f'SMS vs Cuộc gọi (r={cell_corr.iloc[0,1]:.3f})', fontweight='bold')
axes[0].grid(alpha=0.3)

# SMS vs Internet
axes[1].scatter(cell_activity['total_sms'], cell_activity['total_internet'],
                alpha=0.4, s=15, color='chocolate')
axes[1].set_xlabel('Tổng SMS')
axes[1].set_ylabel('Tổng Internet')
axes[1].set_title(f'SMS vs Internet (r={cell_corr.iloc[0,2]:.3f})', fontweight='bold')
axes[1].grid(alpha=0.3)

# Calls vs Internet
axes[2].scatter(cell_activity['total_calls'], cell_activity['total_internet'],
                alpha=0.4, s=15, color='seagreen')
axes[2].set_xlabel('Tổng Cuộc gọi')
axes[2].set_ylabel('Tổng Internet')
axes[2].set_title(f'Cuộc gọi vs Internet (r={cell_corr.iloc[1,2]:.3f})', fontweight='bold')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Insight từ phân tích tương quan:
- Có mối tương quan **rất cao** giữa SMS và Cuộc gọi ở cấp độ ô lưới (r ≈ 0.99). Điều này có nghĩa: các ô lưới có lưu lượng SMS cao thì cũng có lưu lượng cuộc gọi cao.
- Tương quan giữa SMS/Cuộc gọi với Internet cũng rất cao (≈0.97-0.99).
- Ý nghĩa: Mật độ dân cư và mức độ sử dụng dịch vụ viễn thông tại từng khu vực quyết định đồng thời tất cả các loại hoạt động.
- **Hệ quả cho modeling**: Có thể dự đoán một loại hoạt động từ loại khác với độ chính xác cao.

---
## Phần D: Xây dựng mô hình dự đoán

## D.1. Import thư viện sklearn

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, explained_variance_score
import joblib

print("✅ Đã import thư viện sklearn và joblib")

## D.2. Thử nghiệm 1: Dự đoán SMS đi (smsout) từ nhiều đặc trưng

Đây là thử nghiệm gốc từ notebook Kaggle: sử dụng các cột `CellID`, `internet`, `smsin`, `countrycode`, `callin`, `callout` để dự đoán `smsout`. Cách tiếp cận này dùng **dropna** (loại bỏ dòng có NaN) — đây là cách xử lý đơn giản nhưng làm mất nhiều dữ liệu.

### Tại sao không nên dùng dropna?
Như đã thấy ở phần C.4, hơn 87% số dòng có ít nhất một NaN. Dùng dropna sẽ làm mất phần lớn dữ liệu. Tuy nhiên, ta vẫn thực hiện thử nghiệm này để so sánh với phương pháp fillna sau đó.

In [ ]:
# Chuẩn bị dữ liệu theo phương pháp dropna (giữ nguyên thử nghiệm gốc)
print(f"Dữ liệu gốc: {len(combined_df):,} dòng")

df_dropped = combined_df.dropna(axis=0, how='any')
print(f"Sau dropna:   {len(df_dropped):,} dòng")
print(f"Số dòng bị loại bỏ: {len(combined_df) - len(df_dropped):,} dòng "
      f"({(1 - len(df_dropped)/len(combined_df))*100:.1f}%)")

In [ ]:
# Chọn features và target cho Thử nghiệm 1
X_exp1 = df_dropped[['CellID', 'internet', 'smsin', 'countrycode', 'callin', 'callout']]
y_exp1 = df_dropped['smsout']

print(f"Kích thước X: {X_exp1.shape}")
print(f"Kích thước y: {y_exp1.shape}")
print(f"\nCác features:")
for col in X_exp1.columns:
    print(f"  - {col}")

In [ ]:
# Chia train/test (giữ nguyên tỷ lệ gốc: 60/40)
X_train_1, X_test_1, y_train_1, y_test_1 = train_test_split(
    X_exp1, y_exp1, test_size=0.4, random_state=101
)
print(f"Train: {X_train_1.shape[0]:,} mẫu")
print(f"Test:  {X_test_1.shape[0]:,} mẫu")

In [ ]:
# Huấn luyện mô hình Linear Regression
lm_exp1 = LinearRegression()
lm_exp1.fit(X_train_1, y_train_1)

print(f"Hệ số chặn (intercept): {lm_exp1.intercept_:.6f}")
print(f"\nHệ số các biến:")
for name, coef in zip(X_exp1.columns, lm_exp1.coef_):
    print(f"  {name:15s}: {coef:10.6f}")

In [ ]:
# Hệ số dưới dạng DataFrame
coeff_df_1 = pd.DataFrame({'Feature': X_exp1.columns, 'Coefficient': lm_exp1.coef_})
coeff_df_1 = coeff_df_1.sort_values('Coefficient', key=abs, ascending=False)
print("Hệ số hồi quy (đã sắp xếp theo |độ lớn|):")
coeff_df_1

### Nhận xét về hệ số:
- **smsin** có hệ số dương lớn nhất (≈0.73) — SMS đến là predictor mạnh nhất cho SMS đi, điều này trực quan: người nhận SMS thường trả lời lại.
- **callin** có hệ số âm (-0.13) — khi có nhiều cuộc gọi đến, SMS đi giảm (có thể người dùng gọi lại thay vì nhắn tin).
- **internet** và **callout** có hệ số dương nhỏ.
- **countrycode** có hệ số âm rất nhỏ — ảnh hưởng không đáng kể.
- **CellID** có hệ số dương rất nhỏ — vị trí địa lý ít ảnh hưởng đến lượng SMS đi.

In [ ]:
# Dự đoán và đánh giá Thử nghiệm 1
predictions_1 = lm_exp1.predict(X_test_1)

print("="*50)
print("ĐÁNH GIÁ MÔ HÌNH - THỬ NGHIỆM 1")
print("="*50)
print(f"MAE:                   {mean_absolute_error(y_test_1, predictions_1):.4f}")
print(f"MSE:                   {mean_squared_error(y_test_1, predictions_1):.4f}")
print(f"RMSE:                  {np.sqrt(mean_squared_error(y_test_1, predictions_1)):.4f}")
print(f"R² Score:              {r2_score(y_test_1, predictions_1):.6f}")
print(f"Explained Variance:    {explained_variance_score(y_test_1, predictions_1):.6f}")

In [ ]:
# BIỂU ĐỒ 8: Trực quan hóa kết quả Thử nghiệm 1
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Scatter: Thực tế vs Dự đoán
axes[0].scatter(y_test_1, predictions_1, alpha=0.3, s=10, color='steelblue')
max_val = max(y_test_1.max(), predictions_1.max())
axes[0].plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Dự đoán hoàn hảo')
axes[0].set_xlabel('Giá trị thực tế (smsout)')
axes[0].set_ylabel('Giá trị dự đoán (smsout)')
axes[0].set_title('Thực tế vs Dự đoán - TN1', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Residuals histogram
errors_1 = y_test_1 - predictions_1
axes[1].hist(errors_1, bins=50, color='chocolate', edgecolor='white', alpha=0.8)
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Sai số (thực tế - dự đoán)')
axes[1].set_ylabel('Số lượng mẫu')
axes[1].set_title('Phân bố sai số - TN1', fontweight='bold')

# Residuals vs Predicted
axes[2].scatter(predictions_1, errors_1, alpha=0.3, s=10, color='seagreen')
axes[2].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[2].set_xlabel('Giá trị dự đoán')
axes[2].set_ylabel('Sai số')
axes[2].set_title('Sai số theo giá trị dự đoán - TN1', fontweight='bold')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Nhận xét Thử nghiệm 1:
- R² ≈ 0.70: Mô hình giải thích được khoảng 70% phương sai của smsout.
- Sai số tập trung quanh 0 với phân bố gần chuẩn.
- Có hiện tượng phương sai không đồng nhất (heteroscedasticity): sai số tăng khi giá trị dự đoán tăng.
- Mô hình dùng **dropna** nên chỉ học từ ~200K-500K dòng (tùy vào ngày dữ liệu), bỏ phí lượng lớn dữ liệu.

## D.3. Thử nghiệm 2: Dự đoán tổng cuộc gọi từ tổng SMS

Ở phần C.9, ta đã phát hiện mối tương quan rất mạnh giữa `total_sms` và `total_calls` ở cấp ô lưới (r ≈ 0.99). Điều này gợi ý xây dựng mô hình Linear Regression để dự đoán `total_calls` từ `total_sms`.

Trong thử nghiệm này, ta sẽ:
- Sử dụng dữ liệu đã điền missing values (fillna) — tận dụng toàn bộ dữ liệu.
- Tổng hợp theo `CellID` để có một quan sát trên mỗi ô lưới.
- Chia train/val/test (70/15/15).
- Huấn luyện, đánh giá, lưu và load mô hình.

In [ ]:
# Chuẩn bị dữ liệu cho Thử nghiệm 2
# Sử dụng cell_activity đã tính ở phần C.9
X_exp2 = cell_activity[['total_sms']]
y_exp2 = cell_activity['total_calls']

print(f"Kích thước X: {X_exp2.shape}")
print(f"Kích thước y: {y_exp2.shape}")
print(f"\nThống kê mô tả của y (total_calls):")
y_exp2.describe()

In [ ]:
# BIỂU ĐỒ 9: Phân bố biến mục tiêu và mối quan hệ với biến đầu vào
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram của total_calls
axes[0].hist(y_exp2, bins=50, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title('Phân bố của total_calls theo ô lưới', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Tổng số cuộc gọi')
axes[0].set_ylabel('Số lượng ô lưới')

# Scatter total_sms vs total_calls
axes[1].scatter(X_exp2, y_exp2, alpha=0.3, s=15, color='chocolate')
axes[1].set_title('total_sms vs total_calls', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Tổng SMS')
axes[1].set_ylabel('Tổng Cuộc gọi')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## D.4. Chia tập train / validation / test

Chia dữ liệu thành 3 tập với tỷ lệ 70% train, 15% validation, 15% test.
- **Tập train**: dùng để huấn luyện mô hình.
- **Tập validation**: dùng để đánh giá trong quá trình phát triển, chọn tham số.
- **Tập test**: chỉ dùng một lần cuối cùng để đánh giá khả năng tổng quát hóa.

In [ ]:
# Bước 1: Tách 70% train, 30% tạm (val + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X_exp2, y_exp2, test_size=0.30, random_state=42
)

# Bước 2: Tách 30% tạm thành 15% val và 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

print(f"Tập Train:      {X_train.shape[0]:,} mẫu ({X_train.shape[0]/len(X_exp2)*100:.0f}%)")
print(f"Tập Validation: {X_val.shape[0]:,} mẫu ({X_val.shape[0]/len(X_exp2)*100:.0f}%)")
print(f"Tập Test:       {X_test.shape[0]:,} mẫu ({X_test.shape[0]/len(X_exp2)*100:.0f}%)")

In [ ]:
# BIỂU ĐỒ 10: Kiểm tra phân bố của y trên các tập
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (y_data, title) in zip(
    axes,
    [(y_train, 'Train'), (y_val, 'Validation'), (y_test, 'Test')]
):
    ax.hist(y_data, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(f'Phân bố total_calls - Tập {title}', fontweight='bold')
    ax.set_xlabel('Tổng số cuộc gọi')
    ax.set_ylabel('Số lượng ô lưới')
    ax.axvline(y_data.mean(), color='red', linestyle='--', linewidth=1.5,
               label=f'Mean: {y_data.mean():.1f}')
    ax.legend()

plt.tight_layout()
plt.show()

### Nhận xét về việc chia dữ liệu:
- Phân bố của y trên cả 3 tập khá tương đồng — việc chia dữ liệu là hợp lý.
- Phân bố lệch phải (right-skewed): phần lớn ô lưới có tổng cuộc gọi thấp, một số ít ô lưới có tổng cuộc gọi rất cao (trung tâm thành phố).

## D.5. Huấn luyện mô hình Linear Regression

In [ ]:
# Khởi tạo và huấn luyện mô hình
model_exp2 = LinearRegression()
model_exp2.fit(X_train, y_train)

# Hệ số của mô hình
print(f"Hệ số góc (w - slope):     {model_exp2.coef_[0]:.6f}")
print(f"Hệ số chặn (b - intercept): {model_exp2.intercept_:.4f}")
print(f"\nPhương trình hồi quy:")
print(f"  total_calls = {model_exp2.coef_[0]:.6f} * total_sms + {model_exp2.intercept_:.4f}")

### Giải thích phương trình:
- **Hệ số góc (w)**: Mỗi đơn vị SMS tăng thêm, số cuộc gọi tăng thêm khoảng w đơn vị.
- **Hệ số chặn (b)**: Khi SMS = 0, số cuộc gọi dự đoán ≈ b. Đây là baseline khi không có hoạt động SMS.
- Với mô hình này: cứ mỗi 1 SMS, dự đoán có khoảng w cuộc gọi tại ô lưới đó.

## D.6. Đánh giá mô hình

Đánh giá mô hình trên cả 3 tập bằng các chỉ số: MSE (Mean Squared Error), MAE (Mean Absolute Error), RMSE (Root Mean Squared Error), và R² Score.

In [ ]:
# Hàm đánh giá mô hình
def evaluate_model(model, X_data, y_data, dataset_name):
    """Đánh giá mô hình và in kết quả."""
    y_pred = model.predict(X_data)
    mse = mean_squared_error(y_data, y_pred)
    mae = mean_absolute_error(y_data, y_pred)
    r2 = r2_score(y_data, y_pred)
    evs = explained_variance_score(y_data, y_pred)

    print(f"--- {dataset_name} ---")
    print(f"  MSE:                 {mse:,.2f}")
    print(f"  MAE:                 {mae:,.2f}")
    print(f"  RMSE:                {np.sqrt(mse):,.2f}")
    print(f"  R²:                  {r2:.6f}")
    print(f"  Explained Variance:  {evs:.6f}")
    print()
    return y_pred, mse, mae, r2

# Đánh giá trên cả 3 tập
y_train_pred, train_mse, train_mae, train_r2 = evaluate_model(model_exp2, X_train, y_train, "Train")
y_val_pred, val_mse, val_mae, val_r2 = evaluate_model(model_exp2, X_val, y_val, "Validation")
y_test_pred, test_mse, test_mae, test_r2 = evaluate_model(model_exp2, X_test, y_test, "Test")

In [ ]:
# BIỂU ĐỒ 11: So sánh R² trên 3 tập
r2_scores = pd.DataFrame({
    'Tập dữ liệu': ['Train', 'Validation', 'Test'],
    'R² Score': [train_r2, val_r2, test_r2]
})

plt.figure(figsize=(8, 5))
bars = plt.bar(r2_scores['Tập dữ liệu'], r2_scores['R² Score'],
               color=['steelblue', 'darkorange', 'seagreen'], edgecolor='black')
plt.title('So sánh R² Score trên các tập dữ liệu - TN2', fontweight='bold', fontsize=14)
plt.xlabel('Tập dữ liệu')
plt.ylabel('R² Score')
plt.ylim(0, 1.05)

for bar, score in zip(bars, r2_scores['R² Score']):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
             f'{score:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=13)

plt.grid(axis='y', alpha=0.3)
plt.show()

### Nhận xét về kết quả Thử nghiệm 2:
- R² trên cả 3 tập gần bằng 1.0 — mô hình dự đoán gần như hoàn hảo.
- Không có dấu hiệu overfitting: R² train ≈ R² val ≈ R² test.
- Mô hình đơn giản (1 biến) nhưng rất hiệu quả nhờ mối tương quan cực mạnh giữa SMS và Cuộc gọi.

## D.7. Lưu mô hình

Lưu cả hai mô hình đã huấn luyện để sử dụng sau này mà không cần train lại.

In [ ]:
# Tạo thư mục checkpoints
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Lưu mô hình Thử nghiệm 1
MODEL_PATH_EXP1 = CHECKPOINT_DIR / "bai1_exp1_smsout_predictor.joblib"
joblib.dump(lm_exp1, MODEL_PATH_EXP1)
print(f"Đã lưu mô hình TN1 tại: {MODEL_PATH_EXP1}")
print(f"Kích thước file: {MODEL_PATH_EXP1.stat().st_size:,} bytes")

# Lưu mô hình Thử nghiệm 2
MODEL_PATH_EXP2 = CHECKPOINT_DIR / "bai1_exp2_calls_from_sms.joblib"
joblib.dump(model_exp2, MODEL_PATH_EXP2)
print(f"\nĐã lưu mô hình TN2 tại: {MODEL_PATH_EXP2}")
print(f"Kích thước file: {MODEL_PATH_EXP2.stat().st_size:,} bytes")

In [ ]:
# Kiểm tra file đã tồn tại
print(f"File TN1 tồn tại: {MODEL_PATH_EXP1.exists()}")
print(f"File TN2 tồn tại: {MODEL_PATH_EXP2.exists()}")

## D.8. Load mô hình và dự đoán

Minh họa quy trình thực tế: load mô hình đã lưu và dự đoán trên dữ liệu mới. Đây là luồng công việc chuẩn trong triển khai ML: `train → evaluate → save → load → inference → visualize`.

In [ ]:
# Load mô hình từ file
loaded_model_exp2 = joblib.load(MODEL_PATH_EXP2)

# Kiểm tra mô hình đã load đúng chưa
print(f"Hệ số góc (w):     {loaded_model_exp2.coef_[0]:.6f}")
print(f"Hệ số chặn (b):   {loaded_model_exp2.intercept_:.4f}")
print(f"\nSo với mô hình gốc:")
print(f"  w gốc = {model_exp2.coef_[0]:.6f}, w load = {loaded_model_exp2.coef_[0]:.6f}")
print(f"  b gốc = {model_exp2.intercept_:.4f}, b load = {loaded_model_exp2.intercept_:.4f}")
print("✅ Mô hình load khớp với mô hình gốc!")

### D.8.1. Dự đoán trên một mẫu nhỏ

In [ ]:
# Lấy 8 mẫu đầu tiên từ tập test để dự đoán thử
sample_X = X_test.head(8)
sample_y_actual = y_test.head(8)

# Dự đoán
sample_y_pred = loaded_model_exp2.predict(sample_X)

# So sánh kết quả thực tế và dự đoán
sample_results = pd.DataFrame({
    'total_sms': sample_X['total_sms'].values,
    'total_calls_thuc_te': sample_y_actual.values,
    'total_calls_du_doan': sample_y_pred,
    'sai_so': sample_y_actual.values - sample_y_pred,
    'sai_so_phan_tram': np.abs((sample_y_actual.values - sample_y_pred) / sample_y_actual.values) * 100
})
sample_results

In [ ]:
# BIỂU ĐỒ 12: So sánh thực tế vs dự đoán cho 8 mẫu
x_labels = range(1, len(sample_results) + 1)

fig, ax = plt.subplots(figsize=(10, 5))
width = 0.35
ax.bar([x - width/2 for x in x_labels], sample_results['total_calls_thuc_te'],
       width, label='Thực tế', color='steelblue', alpha=0.85, edgecolor='black')
ax.bar([x + width/2 for x in x_labels], sample_results['total_calls_du_doan'],
       width, label='Dự đoán', color='chocolate', alpha=0.85, edgecolor='black')
ax.set_xlabel('Mẫu', fontsize=12)
ax.set_ylabel('Tổng số cuộc gọi', fontsize=12)
ax.set_title('So sánh Thực tế vs Dự đoán - 8 mẫu đầu tiên (TN2)', fontweight='bold', fontsize=13)
ax.set_xticks(x_labels)
ax.legend(fontsize=11)
ax.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

### D.8.2. Dự đoán trên toàn bộ tập test và trực quan hóa

In [ ]:
# Dự đoán trên toàn bộ tập test
y_test_pred_all = loaded_model_exp2.predict(X_test)

# Tạo dataframe kết quả
test_results = pd.DataFrame({
    'total_sms': X_test['total_sms'].values,
    'total_calls_thuc_te': y_test.values,
    'total_calls_du_doan': y_test_pred_all,
    'sai_so': y_test.values - y_test_pred_all
})
test_results['sai_so_tuyet_doi'] = np.abs(test_results['sai_so'])
test_results.head(10)

In [ ]:
# Thống kê sai số trên tập test
print("Thống kê sai số trên tập test (TN2):")
print(f"  Sai số tuyệt đối trung bình (MAE): {test_results['sai_so_tuyet_doi'].mean():,.2f}")
print(f"  Sai số tuyệt đối lớn nhất:         {test_results['sai_so_tuyet_doi'].max():,.2f}")
print(f"  Sai số tuyệt đối nhỏ nhất:         {test_results['sai_so_tuyet_doi'].min():,.4f}")
print(f"  Độ lệch chuẩn của sai số:          {test_results['sai_so_tuyet_doi'].std():,.2f}")

### BIỂU ĐỒ 13: Trực quan hóa Thực tế vs Dự đoán

Nếu mô hình hoàn hảo, tất cả các điểm sẽ nằm trên đường chéo (y = x). Đường chéo màu đỏ thể hiện dự đoán hoàn hảo.

In [ ]:
plt.figure(figsize=(10, 8))

# Scatter plot
plt.scatter(test_results['total_calls_thuc_te'], test_results['total_calls_du_doan'],
            alpha=0.3, s=15, color='steelblue', label='Dự đoán')

# Đường chéo hoàn hảo
max_val = max(test_results['total_calls_thuc_te'].max(), test_results['total_calls_du_doan'].max())
plt.plot([0, max_val], [0, max_val], 'r--', linewidth=2.5, label='Dự đoán hoàn hảo (y = x)')

plt.xlabel('Tổng số cuộc gọi - Thực tế', fontsize=12)
plt.ylabel('Tổng số cuộc gọi - Dự đoán', fontsize=12)
plt.title(f'Thực tế vs Dự đoán trên tập Test (TN2)\nR² = {test_r2:.6f}, MAE = {test_mae:,.2f}',
          fontweight='bold', fontsize=14)
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### BIỂU ĐỒ 14: Phân bố sai số

Phân tích sai số giúp phát hiện các vấn đề tiềm ẩn của mô hình: thiên lệch (bias), phương sai không đồng nhất, hoặc outlier.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram của sai số
axes[0].hist(test_results['sai_so'], bins=50, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Sai số = 0')
axes[0].set_title('Phân bố sai số dự đoán trên tập Test (TN2)', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Sai số (thực tế - dự đoán)')
axes[0].set_ylabel('Số lượng mẫu')
axes[0].legend()

# Scatter sai số theo giá trị thực tế
axes[1].scatter(test_results['total_calls_thuc_te'], test_results['sai_so'],
                alpha=0.3, s=15, color='chocolate')
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[1].set_title('Sai số theo giá trị thực tế (TN2)', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Tổng số cuộc gọi - Thực tế')
axes[1].set_ylabel('Sai số (thực tế - dự đoán)')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### BIỂU ĐỒ 15: Đường hồi quy trên dữ liệu test

Vẽ đường hồi quy của mô hình chồng lên dữ liệu test để thấy độ khớp trực quan.

In [ ]:
plt.figure(figsize=(10, 7))

# Dữ liệu test
plt.scatter(X_test, y_test, alpha=0.3, s=18, color='steelblue', label='Dữ liệu Test (thực tế)')

# Đường hồi quy
x_line = np.linspace(X_test['total_sms'].min(), X_test['total_sms'].max(), 100).reshape(-1, 1)
y_line = loaded_model_exp2.predict(pd.DataFrame(x_line, columns=['total_sms']))
plt.plot(x_line, y_line, 'r-', linewidth=2.5,
         label=f'Đường hồi quy: y = {model_exp2.coef_[0]:.4f}x + {model_exp2.intercept_:.2f}')

plt.xlabel('Tổng SMS', fontsize=12)
plt.ylabel('Tổng Cuộc gọi', fontsize=12)
plt.title('Đường hồi quy Linear Regression trên tập Test (TN2)', fontweight='bold', fontsize=14)
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### BIỂU ĐỒ 16: So sánh kết quả hai thử nghiệm

So sánh R² và RMSE giữa hai cách tiếp cận để thấy sự khác biệt.

In [ ]:
# Tổng hợp metrics so sánh
comparison = pd.DataFrame({
    'Thử nghiệm': ['TN1: Dự đoán smsout\n(dropna, nhiều features)',
                    'TN2: Dự đoán total_calls\n(fillna, 1 feature)'],
    'R²': [r2_score(y_test_1, predictions_1), test_r2],
    'MAE': [mean_absolute_error(y_test_1, predictions_1), test_mae],
    'RMSE': [np.sqrt(mean_squared_error(y_test_1, predictions_1)), np.sqrt(test_mse)],
    'Số mẫu train': [f'{X_train_1.shape[0]:,}', f'{X_train.shape[0]:,}'],
    'Số features': [X_exp1.shape[1], 1]
})

print("SO SÁNH HAI THỬ NGHIỆM")
print("="*70)
comparison

In [ ]:
# BIỂU ĐỒ so sánh
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# So sánh R²
bars = axes[0].bar(['Thử nghiệm 1', 'Thử nghiệm 2'], comparison['R²'],
                   color=['steelblue', 'seagreen'], edgecolor='black')
axes[0].set_title('So sánh R² Score', fontweight='bold')
axes[0].set_ylabel('R² Score')
axes[0].set_ylim(0, 1.1)
for bar, val in zip(bars, comparison['R²']):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                 f'{val:.4f}', ha='center', fontweight='bold')

# So sánh RMSE
bars = axes[1].bar(['Thử nghiệm 1', 'Thử nghiệm 2'], comparison['RMSE'],
                   color=['steelblue', 'seagreen'], edgecolor='black')
axes[1].set_title('So sánh RMSE', fontweight='bold')
axes[1].set_ylabel('RMSE')
for bar, val in zip(bars, comparison['RMSE']):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + bar.get_height() * 0.02,
                 f'{val:,.2f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

### Nhận xét so sánh hai thử nghiệm:
- **TN2 (fillna, 1 feature)** có R² vượt trội hơn TN1 vì:
  - Tận dụng toàn bộ dữ liệu (fillna) thay vì loại bỏ 87% dòng (dropna).
  - Tương quan SMS-Cuộc gọi ở cấp ô lưới là cực kỳ mạnh (≈0.99).
  - Bài toán đơn giản hơn: dự đoán tổng hợp (CellID) thay vì từng bản ghi riêng lẻ.
- **TN1 (dropna, nhiều features)** có R² thấp hơn (≈0.70) nhưng vẫn khá tốt: giải thích 70% phương sai của smsout với 6 features.
- Bài học: **Xử lý missing values đúng cách** và **lựa chọn bài toán phù hợp** quan trọng hơn việc thêm nhiều features.

---
## Phần E: Tổng kết

## E.1. Tổng kết bài 1

Trong bài này, chúng ta đã đi qua quy trình phân tích dữ liệu viễn thông cơ bản với Pandas và xây dựng mô hình dự đoán với sklearn:

### Kỹ năng đã thực hành:

- **Đọc và kiểm tra dữ liệu**: Sử dụng `pd.read_csv` để đọc file dữ liệu CDR, kiểm tra kích thước (`shape`), kiểu dữ liệu (`dtypes`, `info`) và cấu trúc.
- **EDA (Exploratory Data Analysis)**: Khám phá phân bố dữ liệu (`describe`, `value_counts`, `nunique`), thống kê mô tả, trực quan hóa với matplotlib và seaborn (heatmap, jointplot, pairplot, lmplot, scatter, bar, pie, hist).
- **Xử lý dữ liệu thiếu**: Phát hiện missing values (`isna().sum()`), so sánh hai phương pháp: dropna (mất 87% dữ liệu) và fillna với mean (giữ toàn bộ dữ liệu).
- **Tạo đặc trưng mới**: Trích xuất `date`, `hour`, `time` từ cột `datetime`; tạo cột tổng hợp `total_sms`, `total_calls`, `total_internet`.
- **Phân tích theo thời gian**: Xác định giờ cao điểm (17h-19h) và giờ thấp điểm (3h-5h); so sánh hoạt động ban ngày (~73%) và ban đêm (~27%).
- **Phân tích cuộc gọi trong nước và quốc tế**: So sánh mẫu hình theo giờ, tỷ lệ cuộc gọi/SMS trong nước và quốc tế, tỷ lệ cuộc gọi quốc tế đến/đi.
- **Phân tích tương quan**: Tính `corr()` và vẽ heatmap; phát hiện tương quan rất mạnh giữa SMS và Cuộc gọi ở cấp ô lưới (≈0.99).
- **Xây dựng mô hình Linear Regression**: Hai thử nghiệm với cách tiếp cận khác nhau, chia train/val/test (70/15/15), đánh giá bằng MAE, MSE, RMSE, R².
- **Lưu và load mô hình**: Sử dụng `joblib.dump` và `joblib.load` để lưu mô hình vào thư mục checkpoints, load lại và dự đoán trên dữ liệu mới.
- **Trực quan hóa kết quả**: Scatter plot thực tế vs dự đoán, histogram sai số, residual plot, đường hồi quy.

### Các insight chính từ dữ liệu:
1. Hoạt động viễn thông theo nhịp sinh học: cao điểm vào chiều tối, thấp điểm vào rạng sáng.
2. Ban ngày chiếm ~73% tổng lưu lượng, ban đêm ~27%.
3. Ở mỗi ô lưới, các loại hoạt động (SMS, Cuộc gọi, Internet) có tương quan rất mạnh với nhau.
4. Có thể dự đoán chính xác lưu lượng cuộc gọi từ lưu lượng SMS với mô hình tuyến tính đơn giản.
5. Cách xử lý missing values ảnh hưởng lớn đến chất lượng mô hình: fillna tốt hơn dropna với dataset này.

Những kỹ năng này là nền tảng cho các bài học tiếp theo về feature engineering và xây dựng mô hình học máy trên dữ liệu viễn thông.

## E.2. Câu hỏi ôn tập

1. Tại sao việc quan sát thư mục dữ liệu trước khi đọc file là thói quen tốt?
2. Sự khác biệt giữa `df.dropna()` và `df.fillna(df.mean())` là gì? Khi nào nên dùng mỗi phương pháp?
3. Từ ma trận tương quan, làm thế nào để chọn biến đầu vào cho mô hình Linear Regression?
4. Tại sao cần chia dữ liệu thành 3 tập train/val/test thay vì chỉ 2 tập train/test?
5. R² = 0.99 có nghĩa là gì? Có phải lúc nào R² cao cũng là tốt?
6. Làm thế nào để phát hiện overfitting từ kết quả đánh giá mô hình?
7. Tại sao cần lưu mô hình sau khi huấn luyện? Quy trình load và inference diễn ra như thế nào?

## E.3. Bài tập mở rộng

Hãy thử thực hiện các bài tập sau để hiểu sâu hơn:

1. **Thử nghiệm với ngày khác**: Thay vì dùng 3 ngày (02, 04, 06/11), hãy gộp tất cả 7 ngày và xem kết quả mô hình thay đổi như thế nào.

2. **Thử nghiệm với nhiều features hơn**: Trong TN2, thêm `total_internet` làm feature thứ hai. Mô hình có cải thiện không?

3. **Phân tích ô lưới "nóng" và "lạnh"**: Tìm 5 ô lưới có tổng hoạt động cao nhất và thấp nhất. Vẽ biểu đồ so sánh mẫu hình theo giờ của chúng.

4. **Phân tích theo quốc gia cụ thể**: Chọn 3 quốc gia có lưu lượng cao nhất (ngoài Ý) và so sánh mẫu hình hoạt động theo giờ giữa chúng.

5. **Thử nghiệm các phương pháp xử lý missing khác**: So sánh mean imputation với median imputation và constant (0) imputation. Phương pháp nào cho kết quả tốt nhất?